Calculate the lengthscales from the IC

This notebook does:

1. Define the lengthscale functions.
2. Calculate the lengthscales off the IC.

In [2]:
# -------------------------
# Just some imports
# -------------------------

from pathlib import Path
import xarray as xr
import numpy as np

import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

In [3]:
# -------------------------
# Some functions
# -------------------------

def _fft2(a):
    return jnp.fft.fftn(a, axes=(-2, -1))

def _ifft2(a_hat):
    return jnp.fft.ifftn(a_hat, axes=(-2, -1))

def compute_ell_from_w(w, dx, dy, dealias_23=False):
    """
    w: (Ny, Nx) vorticity snapshot (real)
    dx, dy: grid spacings
    dealias_23: apply simple 2/3-rule sharp spectral filter to all spectral fields
    returns: scalar ℓ
    """
    w = jnp.asarray(w, dtype=jnp.float64)
    Ny, Nx = w.shape

    kx = 2.0 * jnp.pi * jnp.fft.fftfreq(Nx, d=dx)  # (Nx,)
    ky = 2.0 * jnp.pi * jnp.fft.fftfreq(Ny, d=dy)  # (Ny,)

    KX, KY = jnp.meshgrid(kx, ky, indexing="xy")   # both (Ny, Nx)
    K2 = KX**2 + KY**2

    if dealias_23:
        kx_cut = (2.0/3.0) * jnp.max(jnp.abs(kx))
        ky_cut = (2.0/3.0) * jnp.max(jnp.abs(ky))
        mask = (jnp.abs(KX) <= kx_cut) & (jnp.abs(KY) <= ky_cut)
        mask = mask.astype(jnp.float64)
    else:
        mask = 1.0

    w_hat = _fft2(w) * mask
    K2_safe = jnp.where(K2 == 0.0, 1.0, K2)
    psi_hat = -w_hat / K2_safe
    psi_hat = psi_hat.at[0, 0].set(0.0 + 0.0j)
    u_hat = (1j * KY) * psi_hat
    v_hat = (-1j * KX) * psi_hat


    ux_hat = (1j * KX) * u_hat
    uy_hat = (1j * KY) * u_hat
    vx_hat = (1j * KX) * v_hat
    vy_hat = (1j * KY) * v_hat

    ux = _ifft2(ux_hat).real
    uy = _ifft2(uy_hat).real
    vx = _ifft2(vx_hat).real
    vy = _ifft2(vy_hat).real

    grad_sq = ux**2 + uy**2 + vx**2 + vy**2
    F1 = jnp.sqrt(jnp.mean(((1.0/12.0)**2) * grad_sq))

    uxx_hat = -(KX**2) * u_hat
    uyy_hat = -(KY**2) * u_hat
    uxy_hat = -(KX * KY) * u_hat

    vxx_hat = -(KX**2) * v_hat
    vyy_hat = -(KY**2) * v_hat
    vxy_hat = -(KX * KY) * v_hat

    uxx = _ifft2(uxx_hat).real
    uyy = _ifft2(uyy_hat).real
    uxy = _ifft2(uxy_hat).real

    vxx = _ifft2(vxx_hat).real
    vyy = _ifft2(vyy_hat).real
    vxy = _ifft2(vxy_hat).real

    second_sq = (uxx**2 + uyy**2 + 2.0*uxy**2) + (vxx**2 + vyy**2 + 2.0*vxy**2)
    F2 = jnp.sqrt(jnp.mean(((1.0/288.0)**2) * second_sq))

    ell = jnp.sqrt(F1 / F2)
    return ell

In [ ]:
# -------------------------
# Load files and run
# -------------------------
path = ".../F1_IC.nc"
ds = xr.open_dataset(path)

w0 = ds["w"].values
Ny, Nx = w0.shape
dx = float(ds.attrs.get("dx", 2*np.pi/Nx))
dy = float(ds.attrs.get("dy", 2*np.pi/Ny))
ell = compute_ell_from_w(w0, dx=dx, dy=dy, dealias_23=False)

print("Ny,Nx =", Ny, Nx)
print("dx,dy =", dx, dy)
print("ℓ =", float(ell))